# Emotion Prediction — Inference Tests
**Author:** Aye Khin Khin Hpone · ST125970 (Yolanda Lim) | Asian Institute of Technology

This notebook validates `model.pkl` works correctly on unseen examples.

Run after `train_pipeline.py` to confirm the serialised model behaves as expected.

In [1]:
import joblib, re, nltk, emoji
import numpy as np
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

for r in ['stopwords','wordnet','omw-1.4']:
    try: nltk.data.find(f'corpora/{r}')
    except: nltk.download(r, quiet=True)

# Negation words preserved — must match train_pipeline.py & app.py exactly
NEGATION_WORDS = {
    "not", "never", "no", "nor", "neither", "nothing", "nobody",
    "nowhere", "without", "very", "extremely", "barely", "hardly"
}

STOP_WORDS = set(stopwords.words('english')) - NEGATION_WORDS
LEMMATIZER = WordNetLemmatizer()
LABEL_MAP  = {0:'Sadness',1:'Joy',2:'Love',3:'Anger',4:'Fear',5:'Surprise'}
CHAT_WORDS = {
    'u':'you','r':'are','ur':'your','lol':'laugh out loud',
    'omg':'oh my god','brb':'be right back','btw':'by the way',
    'idk':'i do not know','imo':'in my opinion','tbh':'to be honest',
    'ngl':'not gonna lie','smh':'shaking my head','ikr':'i know right',
    'nvm':'never mind','gonna':'going to','wanna':'want to',
    'gotta':'got to','kinda':'kind of','cuz':'because','bc':'because',
    'thx':'thanks','ty':'thank you','np':'no problem',
    'asap':'as soon as possible','irl':'in real life',
    'dm':'direct message','gr8':'great','luv':'love',
    'plz':'please','pls':'please','rn':'right now',
}

def clean_text(text):
    """8-step NLP pipeline — must mirror train_pipeline.py & app.py."""
    text = str(text).lower()                           # 1. Lowercase
    text = re.sub(r'\s+', ' ', text).strip()           # 2. Whitespace stripping
    text = re.sub(r'http\S+|www\S+','',text)           # 3. URL removal
    text = emoji.replace_emoji(text,replace='')        # 4. Emoji removal
    text = re.sub(r'[^a-z\s]','',text)                 # 5. Special char removal
    words = text.split()
    words = [CHAT_WORDS.get(w,w) for w in words]       # 6. Chat word expansion
    words = [w for w in words if w not in STOP_WORDS]   # 7. Stopword removal
    words = [LEMMATIZER.lemmatize(w) for w in words]    # 8. Lemmatisation
    return ' '.join(words)

pipeline = joblib.load('model.pkl')
print('Model loaded:', type(pipeline))

c:\Users\ASUS\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator TfidfTransformer from version 1.4.2 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


Model loaded: <class 'sklearn.pipeline.Pipeline'>


c:\Users\ASUS\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator TfidfVectorizer from version 1.4.2 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\ASUS\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator SVC from version 1.4.2 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\ASUS\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator Pipeline f

In [2]:
# ── 6 test cases (one per class) ──────────────────────────────────────────────
test_cases = [
    ('I am so sad and heartbroken today.',                       'Sadness'),
    ('This is the happiest day of my life! I feel amazing!',     'Joy'),
    ('I love spending quality time with my family so much.',     'Love'),
    ('I am absolutely furious and cannot believe what happened.','Anger'),
    ('I was terrified when I heard that noise in the dark.',     'Fear'),
    ('Wow, I never expected this surprise at all!',              'Surprise'),
]

print(f'{"Text":<55} {"Expected":<10} {"Predicted":<10} {"Conf":<8} OK?')
print('─'*100)
for text, expected in test_cases:
    cleaned   = clean_text(text)
    probs     = pipeline.predict_proba([cleaned])[0]
    pred_id   = int(np.argmax(probs))
    predicted = LABEL_MAP[pred_id]
    conf      = probs[pred_id]*100
    ok        = '✓' if predicted == expected else '✗'
    print(f'{text[:53]:<55} {expected:<10} {predicted:<10} {conf:>5.1f}%  {ok}')

Text                                                    Expected   Predicted  Conf     OK?
────────────────────────────────────────────────────────────────────────────────────────────────────
I am so sad and heartbroken today.                      Sadness    Sadness    100.0%  ✓
This is the happiest day of my life! I feel amazing!    Joy        Surprise    99.5%  ✗
I love spending quality time with my family so much.    Love       Joy         52.7%  ✗
I am absolutely furious and cannot believe what happe   Anger      Anger       99.7%  ✓
I was terrified when I heard that noise in the dark.    Fear       Fear       100.0%  ✓
Wow, I never expected this surprise at all!             Surprise   Anger       55.8%  ✗


In [3]:
# ── batch prediction (simulate Flask /predict route) ─────────────────────────
import pandas as pd
batch = [
    "I can't stop crying, everything feels so hopeless.",
    "Just got promoted! Feeling on top of the world right now!",
    "He scared me so badly I couldn't breathe for a moment.",
]
cleaned_batch = [clean_text(t) for t in batch]
all_probs = pipeline.predict_proba(cleaned_batch)

df_res = pd.DataFrame({
    'text': batch,
    'prediction': [LABEL_MAP[np.argmax(p)] for p in all_probs],
    **{LABEL_MAP[i]: (all_probs[:,i]*100).round(2) for i in range(6)},
})
df_res

,text,prediction,Sadness,Joy,Love,Anger,Fear,Surprise
0,"I can't stop crying, everything feels so hopel...",Sadness,99.91,0.06,0.00,0.01,0.02,0.00
1,Just got promoted! Feeling on top of the world...,Joy,7.82,90.55,0.91,0.20,0.23,0.31
2,He scared me so badly I couldn't breathe for a...,Fear,0.38,0.17,0.03,0.61,98.44,0.36
